# CS4603 PA4 — Document Analyst

Development & testing notebook.

- Part 1 / Task 1.7: build the graph, visualize it, run retrieval-only / computation-only / combined queries, show the step-by-step trace.
- Part 2 / Task 2.4: call the deployed endpoint (curl + OpenAI SDK), rerun the same 3 queries, compare local vs. deployed, measure latency.
- Part 3 / Task 3.2: demonstrate `DocumentAnalystClient`. *(pending Part 3)*

## Task 1.7 — Build and visualize the graph

In [1]:
from agent.graph import build_graph

graph = build_graph()
print("Graph built and compiled.")

/Users/saadjamshaidkhan/LUMS/MLops/PA4/cs4603-pa4/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Graph built and compiled.


Processing request of type ListToolsRequest


In [2]:
# Mermaid source for the compiled graph (text form — no external rendering service needed)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	supervisor(supervisor)
	rag_agent(rag_agent)
	mcp_tools(mcp_tools)
	synthesizer(synthesizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	mcp_tools --> supervisor;
	planner --> supervisor;
	rag_agent --> supervisor;
	supervisor -.-> mcp_tools;
	supervisor -.-> rag_agent;
	supervisor -.-> synthesizer;
	synthesizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Test 1 — Retrieval-only query

In [3]:
result = graph.invoke({"messages": [{"role": "user", "content": "What was the net income in 2023?"}]})
print("Plan:", result["plan"])
print("Step results:", result["step_results"])
print("Final answer:", result["final_answer"])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Plan: ['Find the net income of the company for the year 2023']
Step results: ['The net income of the company for the year 2023 was ¥1,107 billion [source: annual_report.pdf, p.1].']
Final answer: The net income in 2023 was ¥1,107 billion [source: annual_report.pdf, p.1].


## Test 2 — Computation-only query

In [4]:
result = graph.invoke({"messages": [{"role": "user", "content": "What is 15% of 2.4 billion?"}]})
print("Plan:", result["plan"])
print("Step results:", result["step_results"])
print("Final answer:", result["final_answer"])

Processing request of type CallToolRequest
Processing request of type ListToolsRequest


Plan: ['Calculate 15% of 2.4 billion']
Step results: ['0.15 * 2.4e9 = 3.6e+08']
Final answer: 15% of 2.4 billion is 0.15 * 2.4e9 = 3.6e+08, or 360 million.


## Test 3 — Combined query (retrieval + computation)

In [5]:
combined_query = "What was the revenue in 2023, and what would a 10% increase look like?"
result = graph.invoke({"messages": [{"role": "user", "content": combined_query}]})
print("Plan:", result["plan"])
print("Step results:", result["step_results"])
print("Final answer:", result["final_answer"])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Processing request of type CallToolRequest
Processing request of type ListToolsRequest


Plan: ['Find the revenue for 2023', 'Calculate a 10% increase on the 2023 revenue found in the previous step']
Step results: ['The revenue for 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1].', '16.91 * 1.10 = 18.601']
Final answer: The revenue in 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1]. A 10% increase would be ¥16.91 trillion * 1.10 = ¥18.601 trillion.


### Step-by-step execution trace for the combined query

Uses `graph.stream(..., stream_mode="updates")` to show exactly which node ran at each step and what it changed in the state.

In [6]:
for i, update in enumerate(
    graph.stream(
        {"messages": [{"role": "user", "content": combined_query}]},
        stream_mode="updates",
    ),
    start=1,
):
    node_name, node_output = next(iter(update.items()))
    print(f"--- step {i}: node = {node_name} ---")
    print(node_output)
    print()

--- step 1: node = planner ---
{'plan': ['Find the revenue for 2023', 'Calculate a 10% increase on the 2023 revenue found in the previous step'], 'current_step_index': 0, 'step_results': []}



--- step 2: node = supervisor ---
{'next_agent': 'rag_agent'}

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


--- step 3: node = rag_agent ---
{'step_results': ['The revenue for 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1].'], 'current_step_index': 1}



--- step 4: node = supervisor ---
{'next_agent': 'mcp_tools'}



Processing request of type CallToolRequest
Processing request of type ListToolsRequest


--- step 5: node = mcp_tools ---
{'step_results': ['The revenue for 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1].', '16.91 * 1.10 = 18.601'], 'current_step_index': 2}

--- step 6: node = supervisor ---
{'next_agent': 'synthesizer'}



--- step 7: node = synthesizer ---
{'final_answer': 'The revenue in 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1]. A 10% increase would be ¥16.91 trillion * 1.10 = ¥18.601 trillion.', 'messages': [AIMessage(content='The revenue in 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1]. A 10% increase would be ¥16.91 trillion * 1.10 = ¥18.601 trillion.', additional_kwargs={}, response_metadata={}, id='842fe81f-c80b-4066-bc6e-c0b52bae2c7c', tool_calls=[], invalid_tool_calls=[])]}



## Task 2.4 — Test the Deployed Endpoint

In [7]:
import os
from config import get_settings

settings = get_settings()
ENDPOINT_NAME = os.environ.get("SERVING_ENDPOINT_NAME", "27100159-document-analyst")
INVOCATION_URL = f"{settings['host']}/serving-endpoints/{ENDPOINT_NAME}/invocations"
print(INVOCATION_URL)

https://dbc-8307750a-af67.cloud.databricks.com/serving-endpoints/27100159-document-analyst/invocations


### 1. Call the endpoint with `curl`

In [8]:
import json as _json

payload = _json.dumps({"messages": [{"role": "user", "content": "What was the net income in 2023?"}]})
with open("/tmp/pa4_payload.json", "w") as f:
    f.write(payload)

!curl -s -X POST "{INVOCATION_URL}" \
  -H "Authorization: Bearer {settings['token']}" \
  -H "Content-Type: application/json" \
  -d @/tmp/pa4_payload.json

[{"messages": [{"content": "What was the net income in 2023?", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": "530d5918-3066-476f-848c-43ef72c7acab"}, {"content": "The net income in 2023 was \u00a51,107 billion [source: annual_report.pdf, p.1].", "additional_kwargs": {}, "response_metadata": {}, "type": "ai", "name": null, "id": "684e28e2-e13a-446e-8f8b-1338c1210d49", "tool_calls": [], "invalid_tool_calls": [], "usage_metadata": null}], "plan": ["Find the net income of the company for the year 2023"], "current_step_index": 1, "step_results": ["The net income of the company for the year 2023 was \u00a51,107 billion [source: annual_report.pdf, p.1]."], "next_agent": "synthesizer", "final_answer": "The net income in 2023 was \u00a51,107 billion [source: annual_report.pdf, p.1]."}]

### 2. Call the endpoint with the OpenAI Python SDK

Our `AnalystState` has extra fields beyond a plain `messages` list (`plan`, `step_results`, `current_step_index`, `next_agent`, `final_answer` — required by Task 1.1), so Databricks' `/chat/completions` gateway route doesn't recognize the output shape as a standard chat completion and leaves `choices=None` on the returned object, instead exposing our own state fields as extra attributes. So we read `state.final_answer` (or `state.messages[-1]["content"]`) instead of `resp.choices[0].message.content` — the call itself still succeeds end-to-end through the OpenAI SDK.

In [9]:
import openai

client = openai.OpenAI(
    api_key=settings["token"],
    base_url=f"{settings['host']}/serving-endpoints",
)

response = client.chat.completions.create(
    model=ENDPOINT_NAME,
    messages=[{"role": "user", "content": "What was the net income in 2023?"}],
)
# response is a list containing one ChatCompletion-shaped object with
# choices=None (see markdown above) — our fields are extra attributes.
state = response[0] if isinstance(response, list) else response
print(state.final_answer)

The net income in 2023 was ¥1,107 billion [source: annual_report.pdf, p.1].


### 3. Rerun the 3 Task 1.7 test queries against the deployed endpoint

In [10]:
def call_deployed(query: str) -> str:
    resp = client.chat.completions.create(
        model=ENDPOINT_NAME,
        messages=[{"role": "user", "content": query}],
    )
    state = resp[0] if isinstance(resp, list) else resp
    return state.final_answer


test_queries = [
    "What was the net income in 2023?",
    "What is 15% of 2.4 billion?",
    "What was the revenue in 2023, and what would a 10% increase look like?",
]

deployed_answers = {}
for q in test_queries:
    answer = call_deployed(q)
    deployed_answers[q] = answer
    print(f"Q: {q}\nA: {answer}\n")

Q: What was the net income in 2023?
A: The net income in 2023 was ¥1,107 billion [source: annual_report.pdf, p.1].



Q: What is 15% of 2.4 billion?
A: 15% of 2.4 billion is 0.15 * 2.4e9 = 3.6e+08, or 360 million.



Q: What was the revenue in 2023, and what would a 10% increase look like?
A: The revenue in 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1]. A 10% increase would be ¥16.91 trillion * 1.10 = ¥18.601 trillion.



### 4. Compare local vs. deployed responses

Rerun the same 3 queries against the local in-process graph (built in Task 1.7) and diff against the deployed answers above.

In [11]:
local_answers = {}
for q in test_queries:
    result = graph.invoke({"messages": [{"role": "user", "content": q}]})
    local_answers[q] = result["final_answer"]

for q in test_queries:
    print(f"Q: {q}")
    print(f"  local:    {local_answers[q]}")
    print(f"  deployed: {deployed_answers[q]}")
    print()

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Processing request of type CallToolRequest
Processing request of type ListToolsRequest


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Processing request of type CallToolRequest
Processing request of type ListToolsRequest


Q: What was the net income in 2023?
  local:    The net income in 2023 was ¥1,107 billion [source: annual_report.pdf, p.1].
  deployed: The net income in 2023 was ¥1,107 billion [source: annual_report.pdf, p.1].

Q: What is 15% of 2.4 billion?
  local:    15% of 2.4 billion is 0.15 * 2.4e9 = 3.6e+08, or 360 million.
  deployed: 15% of 2.4 billion is 0.15 * 2.4e9 = 3.6e+08, or 360 million.

Q: What was the revenue in 2023, and what would a 10% increase look like?
  local:    The revenue in 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1]. A 10% increase would be ¥16.91 trillion * 1.10 = ¥18.601 trillion.
  deployed: The revenue in 2023 was ¥16.91 trillion [source: annual_report.pdf, p.1]. A 10% increase would be ¥16.91 trillion * 1.10 = ¥18.601 trillion.



Both paths run the identical graph and hit the identical Vector Search index and LLM endpoint, so answers should match in substance. Minor wording differences are expected — the LLM calls are not temperature=0 deterministic across processes, and each run is an independent sample.

### 5. Latency: cold start vs. warm

In [12]:
import time

# Cold-start: first request after the endpoint has been idle (scale-to-zero) or
# right after a new deployment. If the endpoint was already warm from the
# calls above, restart the kernel / wait a few minutes before running this
# cell to observe a genuine cold start.
t0 = time.perf_counter()
call_deployed("What was the net income in 2023?")
cold_latency = time.perf_counter() - t0
print(f"Cold-start latency: {cold_latency:.2f}s")

# Warm: immediate follow-up request, container + connections already up.
warm_latencies = []
for _ in range(3):
    t0 = time.perf_counter()
    call_deployed("What was the net income in 2023?")
    warm_latencies.append(time.perf_counter() - t0)

print(f"Warm latencies: {[f'{t:.2f}s' for t in warm_latencies]}")
print(f"Warm avg: {sum(warm_latencies) / len(warm_latencies):.2f}s")

Cold-start latency: 3.74s


Warm latencies: ['4.59s', '3.53s', '7.99s']
Warm avg: 5.37s


## Task 3.2 — Demonstrate the Client SDK

### 1. Import and instantiate `DocumentAnalystClient`

In [13]:
from client.sdk import AnalystClientError, DocumentAnalystClient

sdk_client = DocumentAnalystClient(
    endpoint_name=ENDPOINT_NAME,
    host=settings["host"],
    token=settings["token"],
)
sdk_client

### 2. Health check

In [14]:
is_ready = sdk_client.health_check()
print(f"Endpoint '{ENDPOINT_NAME}' ready: {is_ready}")
assert is_ready, "endpoint is not READY — check `databricks serving-endpoints get <name>`"

Endpoint '27100159-document-analyst' ready: True


### 3. `ask()` with a simple query

In [15]:
answer = sdk_client.ask("What was the net income in 2023?")
print(answer)

The net income in 2023 was ¥1,107 billion [source: annual_report.pdf, p.1].


### 4. `ask_streaming()`

A models-from-code LangChain endpoint may not emit token-by-token `choices[].delta` chunks unless it implements `predict_stream` — see the caveat in Task 3.1. If the endpoint responds with a single complete JSON body instead of an SSE stream, `ask_streaming()` falls back to yielding the full answer once, which is what we expect to see below.

In [16]:
for chunk in sdk_client.ask_streaming("What is 15% of 2.4 billion?"):
    print(chunk, end="", flush=True)
print()

15% of 2.4 billion is 0.15 * 2.4e9 = 3.6e+08, or 360 million.

### 5. Simulate a timeout

Set `timeout=0.001` (1ms) so the request can't possibly complete, and show the client raising a clear `TimeoutError` with the elapsed time rather than hanging or surfacing a raw `httpx` exception.

In [17]:
timeout_client = DocumentAnalystClient(
    endpoint_name=ENDPOINT_NAME,
    host=settings["host"],
    token=settings["token"],
    timeout=0.001,
)

try:
    timeout_client.ask("What was the net income in 2023?")
except TimeoutError as e:
    print(f"Caught expected TimeoutError: {e}")

Caught expected TimeoutError: Request to endpoint '27100159-document-analyst' timed out after 0.01s


### 6. Simulate the endpoint being unavailable (retry behavior)

Point the client at a non-existent endpoint name so every request 404s, and at a local port nothing is listening on so requests come back as connection errors — neither is retryable, so both should surface as a wrapped `AnalystClientError`. To see the exponential-backoff retry path itself (429/503), we monkeypatch `httpx.Client.post` to return 503 twice before succeeding, since we can't make the real deployed endpoint return 503 on demand.

In [18]:
import httpx

unavailable_client = DocumentAnalystClient(
    endpoint_name="this-endpoint-does-not-exist",
    host=settings["host"],
    token=settings["token"],
)

try:
    unavailable_client.ask("What was the net income in 2023?")
except AnalystClientError as e:
    print(f"Caught expected AnalystClientError: {e}")

Caught expected AnalystClientError: The given endpoint does not exist, please retry after checking the specified model and version deployment exists. (status_code=404, request_id=381d564b-78e0-401f-8cbc-56374492cfab)


In [19]:
import time
from unittest.mock import patch

_real_post = httpx.Client.post
_calls = {"n": 0}


def _flaky_post(self, *args, **kwargs):
    _calls["n"] += 1
    if _calls["n"] < 3:
        return httpx.Response(503, json={"message": "endpoint scaling up"}, request=httpx.Request("POST", args[0] if args else kwargs.get("url")))
    return _real_post(self, *args, **kwargs)


t0 = time.perf_counter()
with patch.object(httpx.Client, "post", _flaky_post):
    retry_answer = sdk_client.ask("What was the net income in 2023?")
elapsed = time.perf_counter() - t0

print(f"Succeeded after {_calls['n']} attempts ({elapsed:.1f}s of backoff): {retry_answer}")

Succeeded after 3 attempts (7.4s of backoff): The net income in 2023 was ¥1,107 billion [source: annual_report.pdf, p.1].
